# 17 — Robustness and sensitivity

Test whether conclusions depend on the event definition, transition-year treatment, product definition or source.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

from portugal_refining_resilience.metrics import event_window_summary


In [ ]:
panel = pd.read_csv(PATHS.processed / "fuel_annual_analytical_panel.csv")
rows = []
for event_year in [2021, 2022]:
    for pre_years, post_years in [(3, 2), (5, 3), (8, 3)]:
        for metric in ["exports_kt", "net_import_dependence", "domestic_output_coverage"]:
            if metric not in panel.columns:
                continue
            summary = event_window_summary(panel, value_column=metric, event_year=event_year, pre_years=pre_years, post_years=post_years)
            summary["pre_years"] = pre_years
            summary["post_years"] = post_years
            rows.append(summary)
robust = pd.concat(rows, ignore_index=True)
persist_dataframe(robust, PATHS.metrics / "event_window_sensitivity.csv")
display(robust.head(20))


In [ ]:
# Source reconciliation hook: compare JODI annual trade with a DGEG-extracted canonical file when available.
dgeg_path = PATHS.interim / "dgeg_trade_annual_canonical.csv"
if dgeg_path.exists():
    dgeg = pd.read_csv(dgeg_path)
    jodi = pd.read_csv(PATHS.processed / "fuel_trade_annual.csv")
    comparison = jodi.merge(dgeg, on=["year", "product", "flow"], how="inner", suffixes=("_jodi", "_dgeg"))
    comparison["difference_kt"] = comparison["value_kt_jodi"] - comparison["value_kt_dgeg"]
    comparison["difference_pct_dgeg"] = 100 * comparison["difference_kt"] / comparison["value_kt_dgeg"].replace(0, np.nan)
    persist_dataframe(comparison, PATHS.metrics / "jodi_dgeg_trade_reconciliation.csv", key_columns=["year", "product", "flow"])
    display(comparison.groupby(["product", "flow"])["difference_pct_dgeg"].describe())
else:
    print("DGEG canonical trade extraction not yet present; source reconciliation remains incomplete.")
